In [4]:
import pandas as pd

df = pd.read_csv("../data/superReady_sushi.csv")

columns = df.columns.tolist()

print(columns)

['Unnamed: 0', 'report_month', 'month', 'year', 'sector', 'project_id', 'implementing_agency', 'state', 'date_of_approval', 'approval_year', 'project_age', 'original_cost_crore', 'revised_cost_crore', 'anticipated_cost_crore', 'cumulative_expenditure_crore', 'original_commissioning_date', 'revised_commissioning_date', 'anticipated_commissioning_date', 'delay_revised_months', 'milestones_achieved', 'milestones_total', 'cost_revision_percentage', 'anticipated_cost_percentage', 'exp_vs_anti_prct', 'exp_vs_org_prct', 'exp_vs_rev_prct', 'remaining_org_months', 'revison_delay_months', 'anticipated_delay_from_original', 'delay_revisied_months', 'original_duration_months', 'project_duration_elapsed_percentage', 'milestone_completion_percentage', 'current_cost', 'cost_change_3m', 'cost_growth_3m', 'cost_change_6m', 'cost_growth_6m', 'cost_change_12m', 'cost_growth_12m', 'anticipated_date_3m_ago', 'anticipated_date_6m_ago', 'anticipated_date_12m_ago', 'schedule_change_3m', 'schedule_change_6m', 

In [5]:
import pandas as pd
import numpy as np

# Make sure dates are datetime
df["report_month"] = pd.to_datetime(df["report_month"], errors="coerce")
df["anticipated_commissioning_date"] = pd.to_datetime(
    df["anticipated_commissioning_date"],
    errors="coerce"
)

# Sort project history chronologically
df = df.sort_values(
    ["project_id", "report_month"]
).reset_index(drop=True)

# Create lookup table containing the anticipated date
lookup = df[
    ["project_id", "report_month", "anticipated_commissioning_date"]
].copy()


# ============================================================
# 3 MONTHS AGO
# ============================================================

temp = lookup.copy()

# Move historical observation forward by 3 months
temp["report_month"] = (
    temp["report_month"] + pd.DateOffset(months=3)
)

temp = temp.rename(
    columns={
        "anticipated_commissioning_date":
        "anticipated_date_3m_ago"
    }
)

df = df.merge(
    temp[
        ["project_id", "report_month", "anticipated_date_3m_ago"]
    ],
    on=["project_id", "report_month"],
    how="left"
)


# ============================================================
# 6 MONTHS AGO
# ============================================================

temp = lookup.copy()

temp["report_month"] = (
    temp["report_month"] + pd.DateOffset(months=6)
)

temp = temp.rename(
    columns={
        "anticipated_commissioning_date":
        "anticipated_date_6m_ago"
    }
)

df = df.merge(
    temp[
        ["project_id", "report_month", "anticipated_date_6m_ago"]
    ],
    on=["project_id", "report_month"],
    how="left"
)


# ============================================================
# 12 MONTHS AGO
# ============================================================

temp = lookup.copy()

temp["report_month"] = (
    temp["report_month"] + pd.DateOffset(months=12)
)

temp = temp.rename(
    columns={
        "anticipated_commissioning_date":
        "anticipated_date_12m_ago"
    }
)

df = df.merge(
    temp[
        ["project_id", "report_month", "anticipated_date_12m_ago"]
    ],
    on=["project_id", "report_month"],
    how="left"
)


# ============================================================
# CALCULATE SCHEDULE MOVEMENT IN MONTHS
# ============================================================

def months_difference(later, earlier):
    return (
        (later.dt.year - earlier.dt.year) * 12
        + (later.dt.month - earlier.dt.month)
    )


df["schedule_change_3m"] = months_difference(
    df["anticipated_commissioning_date"],
    df["anticipated_date_3m_ago"]
)

df["schedule_change_6m"] = months_difference(
    df["anticipated_commissioning_date"],
    df["anticipated_date_6m_ago"]
)

df["schedule_change_12m"] = months_difference(
    df["anticipated_commissioning_date"],
    df["anticipated_date_12m_ago"]
)

KeyError: 'anticipated_date_3m_ago'

In [ ]:
df["current_commissioning_date"] = (
    df["anticipated_commissioning_date"]
    .fillna(df["revised_commissioning_date"])
    .fillna(df["original_commissioning_date"])
)

In [ ]:
df.head()


In [ ]:
print(
    df[
        [
            "schedule_change_3m",
            "schedule_change_6m",
            "schedule_change_12m"
        ]
    ].isna().sum()
)

In [ ]:
df[df['schedule_progress_mismatch'] < 0]['milestones_total'].describe()
df[df['schedule_progress_mismatch'] >= 0]['milestones_total'].describe()
# compare distributions

In [ ]:
df['delay_difference_check'] = (
    pd.to_numeric(df['delay_revised_months'], errors='coerce')
    -
    pd.to_numeric(df['delay_revisied_months'], errors='coerce')
)

print(df['delay_difference_check'].describe())

In [ ]:
df[df['schedule_progress_mismatch'] < -20].groupby(
    pd.cut(df['project_duration_elapsed_percentage'], bins=[0,25,50,75,100,1000])
).size()

In [ ]:
# Flag rows where the negative mismatch is likely a milestone-granularity artifact,
# not real schedule performance
df['early_phase_milestone_artifact'] = (
    (df['project_duration_elapsed_percentage'] <= 25) &
    (df['schedule_progress_mismatch'] < -20)
)

In [ ]:
df.head()

In [ ]:
project_level = (
    df.sort_values(["project_id", "report_month"])
      .groupby("project_id")
      .agg(
          implementing_agency=("implementing_agency", "last"),
          sector=("sector", "last"),
          last_report_month=("report_month", "max"),
          final_anticipated_cost=("anticipated_cost_crore", "last"),
          original_cost=("original_cost_crore", "first"),
          final_anticipated_date=(
              "anticipated_commissioning_date",
              "last"
          )
      )
      .reset_index()
)

In [ ]:
project_level["project_had_cost_overrun"] = np.nan

valid = (
    project_level["original_cost"].notna()
    &
    project_level["final_anticipated_cost"].notna()
    &
    (project_level["original_cost"] > 0)
)

project_level.loc[valid, "project_had_cost_overrun"] = (
    (
        project_level.loc[valid, "final_anticipated_cost"]
        -
        project_level.loc[valid, "original_cost"]
    )
    /
    project_level.loc[valid, "original_cost"]
    >= 0.20
).astype(int)

In [ ]:
project_level.shape

In [ ]:
project_level["final_cost_change_pct"] = (
    (
        project_level["final_anticipated_cost"]
        - project_level["original_cost"]
    )
    / project_level["original_cost"]
) * 100

print(
    project_level["final_cost_change_pct"].describe(
        percentiles=[0.10, 0.25, 0.50, 0.75, 0.90, 0.95]
    )
)

In [ ]:
thresholds = [5, 10, 20, 25, 50, 75, 100]

for threshold in thresholds:
    count = (
        project_level["final_cost_change_pct"] >= threshold
    ).sum()

    total = project_level["final_cost_change_pct"].notna().sum()

    percentage = (count / total) * 100

    print(
        f"{threshold}% overrun: "
        f"{count} projects ({percentage:.2f}%)"
    )

In [ ]:
double_cost_projects = project_level[
    project_level["final_cost_change_pct"] >= 100
]

print("Projects with >=100% cost increase:")
print(len(double_cost_projects))

valid_projects = project_level[
    project_level["final_cost_change_pct"].notna()
]

print(
    "Percentage:",
    len(double_cost_projects) / len(valid_projects) * 100
)

decreased = project_level[
    project_level["final_cost_change_pct"] < 0
]

print("Projects where anticipated cost decreased:")
print(len(decreased))

print(
    "Percentage:",
    len(decreased) / len(valid_projects) * 100
)

In [ ]:
print(
    project_level["project_had_cost_overrun"]
    .value_counts(dropna=False)
)

In [ ]:
df["completion_proxy"] = (
    df["milestones_total"].notna()
    & (df["milestones_total"] > 0)
    & df["milestones_achieved"].notna()
    & (df["milestones_achieved"] >= df["milestones_total"])
).astype(int)

In [ ]:
df.head()

In [ ]:
print(df.columns.tolist())

In [ ]:
print(project_level.columns.tolist())

In [ ]:
import pandas as pd
import numpy as np

# ============================================================
# CROSS-PROJECT BENCHMARK FEATURES
# ============================================================

# ------------------------------------------------------------
# 1. Prepare data types
# ------------------------------------------------------------

df['report_month'] = pd.to_datetime(
    df['report_month'],
    errors='coerce'
)

project_level['completion_date'] = pd.to_datetime(
    project_level['completion_date'],
    errors='coerce'
)

project_level['project_had_cost_overrun'] = pd.to_numeric(
    project_level['project_had_cost_overrun'],
    errors='coerce'
)


# ------------------------------------------------------------
# 2. Historical completed projects
# ------------------------------------------------------------

hist = project_level[
    project_level['completion_date'].notna() &
    project_level['project_had_cost_overrun'].notna()
].copy()


# Overall historical overrun rate
prior_rate = hist['project_had_cost_overrun'].mean()

# Shrinkage parameter
ALPHA = 5


# ============================================================
# 3. Create project-month snapshots
# ============================================================

snap = df[
    [
        'project_id',
        'report_month',
        'implementing_agency',
        'sector'
    ]
].drop_duplicates().copy()


# ============================================================
# 4. AGENCY HISTORICAL STATISTICS
# ============================================================

agency_events = (
    hist[
        [
            'implementing_agency',
            'completion_date',
            'project_had_cost_overrun'
        ]
    ]
    .dropna(
        subset=[
            'implementing_agency',
            'completion_date'
        ]
    )
    .groupby(
        [
            'implementing_agency',
            'completion_date'
        ],
        as_index=False
    )
    .agg(
        event_sum=(
            'project_had_cost_overrun',
            'sum'
        ),
        event_count=(
            'project_had_cost_overrun',
            'count'
        )
    )
)

agency_events = agency_events.sort_values(
    ['completion_date', 'implementing_agency']
).reset_index(drop=True)


# Cumulative statistics
agency_events['cum_sum'] = (
    agency_events
    .groupby('implementing_agency')['event_sum']
    .cumsum()
)

agency_events['cum_count'] = (
    agency_events
    .groupby('implementing_agency')['event_count']
    .cumsum()
)


# Statistics strictly BEFORE that completion date
agency_events['prior_sum'] = (
    agency_events
    .groupby('implementing_agency')['cum_sum']
    .shift(1)
    .fillna(0)
)

agency_events['prior_count'] = (
    agency_events
    .groupby('implementing_agency')['cum_count']
    .shift(1)
    .fillna(0)
)


agency_events = agency_events[
    [
        'implementing_agency',
        'completion_date',
        'prior_sum',
        'prior_count'
    ]
]


# ============================================================
# 5. MERGE AGENCY HISTORY
# ============================================================

snap_agency = snap.copy()

snap_agency = snap_agency.dropna(
    subset=[
        'report_month',
        'implementing_agency'
    ]
)

snap_agency = snap_agency.sort_values(
    ['report_month', 'implementing_agency']
).reset_index(drop=True)

agency_events = agency_events.sort_values(
    ['completion_date', 'implementing_agency']
).reset_index(drop=True)


snap_agency = pd.merge_asof(
    snap_agency,
    agency_events,
    left_on='report_month',
    right_on='completion_date',
    left_by='implementing_agency',
    right_by='implementing_agency',
    direction='backward',
    allow_exact_matches=False
)


# Rename the historical completion date
# so it doesn't cause confusion
snap_agency = snap_agency.rename(
    columns={
        'completion_date': 'last_historical_completion_date'
    }
)


# ------------------------------------------------------------
# IMPORTANT:
# Remove the current project from the benchmark if the
# current project itself has already completed.
# ------------------------------------------------------------

# Get each project's actual completion date separately
project_completion = project_level[
    [
        'project_id',
        'completion_date',
        'project_had_cost_overrun'
    ]
].drop_duplicates('project_id')

project_completion = project_completion.rename(
    columns={
        'completion_date': 'own_completion_date',
        'project_had_cost_overrun': 'own_overrun'
    }
)


snap_agency = snap_agency.merge(
    project_completion,
    on='project_id',
    how='left'
)


current_completed = (
    snap_agency['own_completion_date'].notna() &
    (
        snap_agency['own_completion_date']
        < snap_agency['report_month']
    )
)


snap_agency['agency_prior_sum'] = (
    snap_agency['prior_sum'].fillna(0)
)

snap_agency['agency_prior_count'] = (
    snap_agency['prior_count'].fillna(0)
)


# Remove current project's own result
snap_agency.loc[
    current_completed,
    'agency_prior_sum'
] -= snap_agency.loc[
    current_completed,
    'own_overrun'
].fillna(0)


snap_agency.loc[
    current_completed,
    'agency_prior_count'
] -= 1


snap_agency['agency_prior_count'] = (
    snap_agency['agency_prior_count'].clip(lower=0)
)


# ============================================================
# 6. SECTOR HISTORICAL STATISTICS
# ============================================================

sector_events = (
    hist[
        [
            'sector',
            'completion_date',
            'project_had_cost_overrun'
        ]
    ]
    .dropna(
        subset=[
            'sector',
            'completion_date'
        ]
    )
    .groupby(
        [
            'sector',
            'completion_date'
        ],
        as_index=False
    )
    .agg(
        event_sum=(
            'project_had_cost_overrun',
            'sum'
        ),
        event_count=(
            'project_had_cost_overrun',
            'count'
        )
    )
)


sector_events = sector_events.sort_values(
    ['completion_date', 'sector']
).reset_index(drop=True)


sector_events['cum_sum'] = (
    sector_events
    .groupby('sector')['event_sum']
    .cumsum()
)

sector_events['cum_count'] = (
    sector_events
    .groupby('sector')['event_count']
    .cumsum()
)


sector_events['prior_sum'] = (
    sector_events
    .groupby('sector')['cum_sum']
    .shift(1)
    .fillna(0)
)

sector_events['prior_count'] = (
    sector_events
    .groupby('sector')['cum_count']
    .shift(1)
    .fillna(0)
)


sector_events = sector_events[
    [
        'sector',
        'completion_date',
        'prior_sum',
        'prior_count'
    ]
]


# ============================================================
# 7. MERGE SECTOR HISTORY
# ============================================================

snap_sector = snap.copy()

snap_sector = snap_sector.dropna(
    subset=[
        'report_month',
        'sector'
    ]
)


snap_sector = snap_sector.sort_values(
    ['report_month', 'sector']
).reset_index(drop=True)


sector_events = sector_events.sort_values(
    ['completion_date', 'sector']
).reset_index(drop=True)


snap_sector = pd.merge_asof(
    snap_sector,
    sector_events,
    left_on='report_month',
    right_on='completion_date',
    left_by='sector',
    right_by='sector',
    direction='backward',
    allow_exact_matches=False
)


# ============================================================
# 8. REMOVE CURRENT PROJECT FROM SECTOR BENCHMARK
# ============================================================

snap_sector = snap_sector.rename(
    columns={
        'completion_date':
            'last_historical_completion_date'
    }
)


snap_sector = snap_sector.merge(
    project_completion,
    on='project_id',
    how='left'
)


current_completed = (
    snap_sector['own_completion_date'].notna() &
    (
        snap_sector['own_completion_date']
        < snap_sector['report_month']
    )
)


snap_sector['sector_prior_sum'] = (
    snap_sector['prior_sum'].fillna(0)
)

snap_sector['sector_prior_count'] = (
    snap_sector['prior_count'].fillna(0)
)


snap_sector.loc[
    current_completed,
    'sector_prior_sum'
] -= snap_sector.loc[
    current_completed,
    'own_overrun'
].fillna(0)


snap_sector.loc[
    current_completed,
    'sector_prior_count'
] -= 1


snap_sector['sector_prior_count'] = (
    snap_sector['sector_prior_count'].clip(lower=0)
)


# ============================================================
# 9. CALCULATE SMOOTHED RATES
# ============================================================

snap_agency['agency_overrun_rate'] = (
    snap_agency['agency_prior_sum']
    + ALPHA * prior_rate
) / (
    snap_agency['agency_prior_count']
    + ALPHA
)


snap_sector['sector_overrun_rate'] = (
    snap_sector['sector_prior_sum']
    + ALPHA * prior_rate
) / (
    snap_sector['sector_prior_count']
    + ALPHA
)


# ============================================================
# 10. CREATE FINAL BENCHMARK TABLE
# ============================================================

benchmark_features = snap_agency[
    [
        'project_id',
        'report_month',
        'agency_overrun_rate',
        'agency_prior_count'
    ]
].merge(
    snap_sector[
        [
            'project_id',
            'report_month',
            'sector_overrun_rate'
        ]
    ],
    on=[
        'project_id',
        'report_month'
    ],
    how='outer'
)


benchmark_features = benchmark_features.rename(
    columns={
        'agency_prior_count':
            'agency_project_count_as_of_T'
    }
)


# ============================================================
# 11. REMOVE OLD VERSIONS
# ============================================================

df = df.drop(
    columns=[
        'agency_overrun_rate',
        'sector_overrun_rate',
        'agency_project_count_as_of_T'
    ],
    errors='ignore'
)


# ============================================================
# 12. MERGE FEATURES INTO MAIN DATASET
# ============================================================

df = df.merge(
    benchmark_features,
    on=[
        'project_id',
        'report_month'
    ],
    how='left'
)


# ============================================================
# 13. FINAL CHECK
# ============================================================

print("\n✅ Benchmark features created!\n")

print(
    df[
        [
            'agency_overrun_rate',
            'sector_overrun_rate',
            'agency_project_count_as_of_T'
        ]
    ].describe()
)

print("\nMissing values:")

print(
    df[
        [
            'agency_overrun_rate',
            'sector_overrun_rate',
            'agency_project_count_as_of_T'
        ]
    ].isna().sum()
)

print("\nSample:")

print(
    df[
        [
            'project_id',
            'report_month',
            'implementing_agency',
            'sector',
            'agency_overrun_rate',
            'sector_overrun_rate',
            'agency_project_count_as_of_T'
        ]
    ].head(20)
)

In [ ]:
df.head()

In [ ]:
print("Total rows:", len(df))

print("\nRows with current anticipated date:")
print(df["anticipated_commissioning_date"].notna().sum())

print("\nRows with future anticipated date:")
print(df["future_anticipated_date"].notna().sum())

print("\nReport months:")
print(df["report_month"].min(), "to", df["report_month"].max())

print("\nUnique projects:", df["project_id"].nunique())

report_counts = (
    df.groupby("project_id")["report_month"]
      .count()
)

print(report_counts.describe())

In [ ]:
date_cols = [
    "original_commissioning_date",
    "revised_commissioning_date",
    "anticipated_commissioning_date"
]

print(df[date_cols].notna().sum())
print("\nMissing percentage:")
print(df[date_cols].isna().mean() * 100)

print(
    df[date_cols]
    .notna()
    .value_counts()
)

In [ ]:
print(
    "Current anticipated:",
    df["anticipated_commissioning_date"].notna().sum()
)

print(
    "Current revised:",
    df["revised_commissioning_date"].notna().sum()
)

print(
    "Current original:",
    df["original_commissioning_date"].notna().sum()
)

In [ ]:
# Make sure all dates are datetime
date_cols = [
    "original_commissioning_date",
    "revised_commissioning_date",
    "anticipated_commissioning_date"
]

for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors="coerce")


# Best available current expected completion date
df["current_expected_completion"] = (
    df["anticipated_commissioning_date"]
    .combine_first(df["revised_commissioning_date"])
    .combine_first(df["original_commissioning_date"])
)


# Check coverage
print("Total rows:", len(df))

print(
    "Current expected completion available:",
    df["current_expected_completion"].notna().sum()
)

print(
    "Missing:",
    df["current_expected_completion"].isna().sum()
)

print(
    "Coverage:",
    df["current_expected_completion"].notna().mean() * 100,
    "%"
)
print("\nSource of current_expected_completion:")

print(
    pd.Series(
        np.select(
            [
                df["anticipated_commissioning_date"].notna(),
                df["revised_commissioning_date"].notna(),
                df["original_commissioning_date"].notna()
            ],
            [
                "anticipated",
                "revised",
                "original"
            ],
            default="missing"
        )
    ).value_counts()
)

In [ ]:

# ============================================================
# TIME OVERRUN TARGET
# ============================================================
#
# Definition:
#
# At project snapshot T, predict whether the project's
# expected completion date will be pushed back by >= 6 months
# over approximately the next 12 months.
#
# Current completion date hierarchy:
#   1. anticipated_commissioning_date
#   2. revised_commissioning_date
#   3. original_commissioning_date
#
# Future observation:
#   Search between T+10 and T+14 months.
#   Select the observation closest to T+12.
#
# Target:
#   1 = schedule pushed back >= 6 months
#   0 = schedule pushed back < 6 months
#   NaN = future outcome unavailable
# ============================================================


# ------------------------------------------------------------
# 1. Convert dates
# ------------------------------------------------------------

date_cols = [
    "report_month",
    "original_commissioning_date",
    "revised_commissioning_date",
    "anticipated_commissioning_date"
]

for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors="coerce")


# ------------------------------------------------------------
# 2. Best available current expected completion date
# ------------------------------------------------------------

df["current_expected_completion"] = (
    df["anticipated_commissioning_date"]
    .combine_first(df["revised_commissioning_date"])
    .combine_first(df["original_commissioning_date"])
)


# ------------------------------------------------------------
# 3. Create month index
# ------------------------------------------------------------

df["month_index"] = (
    df["report_month"].dt.year * 12
    + df["report_month"].dt.month
)


# ------------------------------------------------------------
# 4. Give every row a unique temporary ID
# ------------------------------------------------------------

df = df.reset_index(drop=True)

df["_row_id"] = np.arange(len(df))


# ------------------------------------------------------------
# 5. Create future lookup
# ------------------------------------------------------------

future_lookup = df[
    [
        "project_id",
        "month_index",
        "current_expected_completion"
    ]
].copy()

future_lookup = future_lookup.rename(columns={
    "month_index": "future_month_index",
    "current_expected_completion": "future_expected_completion"
})


# Only future dates that actually exist
future_lookup = future_lookup[
    future_lookup["future_expected_completion"].notna()
].copy()


# ------------------------------------------------------------
# 6. Generate possible future horizons
# ------------------------------------------------------------

offsets = [10, 11, 12, 13, 14]

candidate_list = []

for offset in offsets:

    temp = df[
        [
            "_row_id",
            "project_id",
            "month_index"
        ]
    ].copy()

    temp["future_month_index"] = (
        temp["month_index"] + offset
    )

    temp["future_horizon_months"] = offset

    candidate_list.append(temp)


candidates = pd.concat(
    candidate_list,
    ignore_index=True
)


# ------------------------------------------------------------
# 7. Match future observations
# ------------------------------------------------------------

candidates = candidates.merge(
    future_lookup,
    on=[
        "project_id",
        "future_month_index"
    ],
    how="left"
)


# ------------------------------------------------------------
# 8. Remove candidates where future completion date
#    is unavailable
# ------------------------------------------------------------

candidates_valid = candidates[
    candidates["future_expected_completion"].notna()
].copy()


# ------------------------------------------------------------
# 9. Find closest observation to 12 months
# ------------------------------------------------------------

candidates_valid["distance_from_12"] = (
    abs(
        candidates_valid["future_horizon_months"] - 12
    )
)


# Sort so that:
#   +12 months is preferred
#   then +11/+13
#   then +10/+14

candidates_valid = candidates_valid.sort_values(
    [
        "_row_id",
        "distance_from_12",
        "future_horizon_months"
    ]
)


# Select exactly ONE future observation per current row
best_future = (
    candidates_valid
    .drop_duplicates(
        subset="_row_id",
        keep="first"
    )
    [
        [
            "_row_id",
            "future_expected_completion",
            "future_horizon_months"
        ]
    ]
)


# ------------------------------------------------------------
# 10. Merge selected future observation back
# ------------------------------------------------------------

df = df.merge(
    best_future,
    on="_row_id",
    how="left"
)


# ------------------------------------------------------------
# 11. Calculate future schedule shift
# ------------------------------------------------------------

df["future_schedule_shift_12m"] = np.nan

valid_shift = (
    df["current_expected_completion"].notna()
    & df["future_expected_completion"].notna()
)

df.loc[valid_shift, "future_schedule_shift_12m"] = (

    (
        df.loc[
            valid_shift,
            "future_expected_completion"
        ].dt.year
        -
        df.loc[
            valid_shift,
            "current_expected_completion"
        ].dt.year
    ) * 12

    +

    (
        df.loc[
            valid_shift,
            "future_expected_completion"
        ].dt.month
        -
        df.loc[
            valid_shift,
            "current_expected_completion"
        ].dt.month
    )
)


# ------------------------------------------------------------
# 12. Create final time-overrun target
# ------------------------------------------------------------

df["target_schedule_risk_12m"] = np.nan

df.loc[valid_shift, "target_schedule_risk_12m"] = (
    df.loc[
        valid_shift,
        "future_schedule_shift_12m"
    ] >= 6
).astype(int)


# ------------------------------------------------------------
# 13. Display results
# ------------------------------------------------------------

print("=" * 60)
print("TIME OVERRUN TARGET RESULTS")
print("=" * 60)

print("\nTotal rows:")
print(len(df))

print("\nCurrent expected completion available:")
print(
    df["current_expected_completion"].notna().sum()
)

print("\nFuture observation available:")
print(
    df["future_expected_completion"].notna().sum()
)

print("\nTarget distribution:")
print(
    df["target_schedule_risk_12m"]
    .value_counts(dropna=False)
)

print("\nTarget percentages:")
print(
    df["target_schedule_risk_12m"]
    .value_counts(normalize=True, dropna=False)
    .mul(100)
    .round(2)
)

print("\nFuture horizon distribution:")
print(
    df["future_horizon_months"]
    .value_counts(dropna=True)
    .sort_index()
)

print("\nSchedule shift statistics:")
print(
    df["future_schedule_shift_12m"]
    .describe()
)

print("\nSchedule shift distribution:")
print(
    df["future_schedule_shift_12m"]
    .value_counts()
    .sort_index()
    .head(40)
)


# ------------------------------------------------------------
# 14. Remove temporary columns
# ------------------------------------------------------------

df.drop(
    columns=[
        "_row_id",
        "month_index"
    ],
    inplace=True,
    errors="ignore"
)

In [ ]:
print(
    df["target_schedule_risk_12m"]
    .value_counts(dropna=False)
)

print(
    df["target_schedule_risk_12m"]
    .value_counts(normalize=True, dropna=False) * 100
)

print(
    df["future_horizon_months"]
    .value_counts(dropna=True).sort_index()
)

print(
    df["future_schedule_shift_12m"].describe()
)

In [ ]:
df["target_schedule_risk_12m"].value_counts()

In [ ]:

# ============================================================
# CURRENT TARGET DIAGNOSTIC
# ============================================================

target_col = "target_schedule_risk_12m"

print("=" * 70)
print("TIME OVERRUN TARGET DIAGNOSTIC")
print("=" * 70)


# ------------------------------------------------------------
# 1. Overall coverage
# ------------------------------------------------------------

total_rows = len(df)

known = df[target_col].notna()
known_rows = known.sum()
unknown_rows = (~known).sum()

print("\n1. LABEL COVERAGE")
print("-" * 50)

print(f"Total rows:        {total_rows:,}")
print(f"Known labels:      {known_rows:,}")
print(f"Unknown labels:    {unknown_rows:,}")
print(f"Coverage:          {known_rows / total_rows * 100:.2f}%")
print(f"Missing:           {unknown_rows / total_rows * 100:.2f}%")


# ------------------------------------------------------------
# 2. Class distribution
# ------------------------------------------------------------

print("\n2. CLASS DISTRIBUTION")
print("-" * 50)

print(
    df[target_col]
    .value_counts(dropna=False)
)

print("\nKnown labels only:")

print(
    df.loc[known, target_col]
    .value_counts()
)

print("\nPercent among known labels:")

print(
    df.loc[known, target_col]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)


# ------------------------------------------------------------
# 3. Unique project coverage
# ------------------------------------------------------------

print("\n3. PROJECT COVERAGE")
print("-" * 50)

total_projects = df["project_id"].nunique()

labelled_projects = df.loc[
    known,
    "project_id"
].nunique()

print(f"Total projects:       {total_projects:,}")
print(f"Projects with labels: {labelled_projects:,}")
print(
    f"Project coverage:     "
    f"{labelled_projects / total_projects * 100:.2f}%"
)


# ------------------------------------------------------------
# 4. Future horizon
# ------------------------------------------------------------

if "future_horizon_months" in df.columns:

    print("\n4. FUTURE HORIZON")
    print("-" * 50)

    print(
        df.loc[known, "future_horizon_months"]
        .value_counts()
        .sort_index()
    )

    print("\nStatistics:")

    print(
        df.loc[
            known,
            "future_horizon_months"
        ].describe()
    )


# ------------------------------------------------------------
# 5. Schedule shift
# ------------------------------------------------------------

if "future_schedule_shift_12m" in df.columns:

    print("\n5. SCHEDULE SHIFT")
    print("-" * 50)

    shift = df.loc[
        known,
        "future_schedule_shift_12m"
    ].dropna()

    print(shift.describe())

    print("\nExtreme negative shifts:")
    print(shift.nsmallest(10).to_list())

    print("\nExtreme positive shifts:")
    print(shift.nlargest(10).to_list())


# ------------------------------------------------------------
# 6. Current expected completion coverage
# ------------------------------------------------------------

if "current_expected_completion" in df.columns:

    print("\n6. CURRENT COMPLETION DATE COVERAGE")
    print("-" * 50)

    print(
        f"Available: "
        f"{df['current_expected_completion'].notna().sum():,}"
    )

    print(
        f"Missing:   "
        f"{df['current_expected_completion'].isna().sum():,}"
    )


# ------------------------------------------------------------
# 7. Label distribution over time
# ------------------------------------------------------------

print("\n7. LABELS BY REPORT YEAR")
print("-" * 50)

if "year" in df.columns:

    yearly = (
        df.loc[known]
        .groupby("year")[target_col]
        .agg(
            rows="count",
            positive="sum"
        )
    )

    yearly["negative"] = (
        yearly["rows"] - yearly["positive"]
    )

    yearly["positive_rate_%"] = (
        yearly["positive"]
        / yearly["rows"]
        * 100
    ).round(2)

    print(yearly)


# ------------------------------------------------------------
# 8. Target by project
# ------------------------------------------------------------

print("\n8. PROJECT-LEVEL TARGET BEHAVIOUR")
print("-" * 50)

project_target = (
    df.loc[known]
    .groupby("project_id")[target_col]
    .agg(
        observations="count",
        positive_count="sum",
        positive_rate="mean"
    )
)

print(
    project_target["positive_rate"]
    .describe()
)

print("\nProjects with 100% positive labels:")
print(
    (project_target["positive_rate"] == 1).sum()
)

print("\nProjects with 0% positive labels:")
print(
    (project_target["positive_rate"] == 0).sum()
)


# ------------------------------------------------------------
# 9. Final summary score
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("SUMMARY")
print("=" * 70)

print(
    f"""
Label coverage:       {known_rows / total_rows * 100:.2f}%
Project coverage:     {labelled_projects / total_projects * 100:.2f}%
Positive rate:        {
    df.loc[known, target_col].mean() * 100
    if known_rows > 0 else np.nan
:.2f}%
"""
)

In [ ]:
df[['current_expected_completion',
   'future_expected_completion',
   'future_schedule_shift_12m',
   'target_schedule_risk_12m']].sort_values(
       'future_schedule_shift_12m'
).head(20)
df[['current_expected_completion',
   'future_expected_completion',
   'future_schedule_shift_12m',
   'target_schedule_risk_12m']].sort_values(
       'future_schedule_shift_12m',
       ascending=False
).head(20)

In [ ]:
print("1999-01-01:", (df["current_expected_completion"] == pd.Timestamp("1999-01-01")).sum())
print("2003-03-01:", (df["current_expected_completion"] == pd.Timestamp("2003-03-01")).sum())

In [ ]:
df[df["current_expected_completion"].isin([
    pd.Timestamp("1999-01-01"),
    pd.Timestamp("2003-03-01")
])][[
    "project_id",
    "project_name",
    "report_month",
    "original_commissioning_date",
    "revised_commissioning_date",
    "anticipated_commissioning_date",
    "current_expected_completion",
    "future_expected_completion",
    "future_schedule_shift_12m"
]].head()

In [ ]:
# Remove clearly unrealistic schedule shifts
df = df[
    df["future_schedule_shift_12m"].between(-60, 60)
    | df["future_schedule_shift_12m"].isna()
].copy()

print("Rows remaining:", len(df))
print("\nTarget distribution:")
print(df["target_schedule_risk_12m"].value_counts(dropna=False))

print("\nSchedule shift statistics:")
print(df["future_schedule_shift_12m"].describe())

In [ ]:
df.head()

In [ ]:
df.to_csv("../data/sushitime.csv")

In [ ]:
print(df.columns.tolist())

In [ ]:
import pandas as pd
import numpy as np

# Make sure all date columns are actually datetime
date_cols = [
    'date_of_approval',
    'report_month',
    'original_commissioning_date'
]

for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')


# ============================================================
# 1. ORIGINAL PROJECT DURATION IN MONTHS
# ============================================================

df['original_duration_months'] = (
    (df['original_commissioning_date'].dt.year -
     df['date_of_approval'].dt.year) * 12
    +
    (df['original_commissioning_date'].dt.month -
     df['date_of_approval'].dt.month)
)

# Invalid durations → NaN
df.loc[df['original_duration_months'] <= 0,
       'original_duration_months'] = np.nan


# ============================================================
# 2. PROJECT AGE AT EACH REPORT MONTH
# ============================================================

df['project_age'] = (
    (df['report_month'].dt.year -
     df['date_of_approval'].dt.year) * 12
    +
    (df['report_month'].dt.month -
     df['date_of_approval'].dt.month)
)

# Negative age doesn't make sense
df.loc[df['project_age'] < 0, 'project_age'] = np.nan


# ============================================================
# 3. PROJECT DURATION ELAPSED %
# ============================================================

df['project_duration_elapsed_percentage'] = (
    df['project_age'] /
    df['original_duration_months']
) * 100

# Keep percentage sensible
df.loc[
    ~np.isfinite(df['project_duration_elapsed_percentage']),
    'project_duration_elapsed_percentage'
] = np.nan


# ============================================================
# CHECK
# ============================================================

print(
    df[
        [
            'date_of_approval',
            'report_month',
            'original_commissioning_date',
            'original_duration_months',
            'project_age',
            'project_duration_elapsed_percentage'
        ]
    ].head(20)
)

In [ ]:
df['duration_overrun_months'] = (
    df['project_age'] - df['original_duration_months']
)

df['duration_overrun_months'] = df['duration_overrun_months'].clip(lower=0)

In [ ]:
df.head(5)

In [ ]:
print(
    df['delay_difference_check']
    .value_counts(dropna=False)
    .head(20)
)

In [ ]:
df.head()

In [ ]:
df.to_csv("../data/sushi_gaanduuu.csv")